In [1]:
import pandas as pd

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"

columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [3]:
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [18]:
from models import Ride,ride_from_row,ride_serializer

In [19]:
ride = ride_from_row(df.iloc[0])
ride

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [13]:
# Kafka works with raw bytes, so we need a serializer that converts Python dicts to JSON
import json
def json_serializer(data):
    return json.dumps(data).encode('utf-8')



In [14]:
from kafka import KafkaProducer
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)


In [15]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

In [16]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

producer.send(topic_name, value=ride)
producer.flush()
